# GPEC 447 Project Notebook: Otay Mesa Industrial–Logistics Coupling

**Project question:**  
Does Otay Mesa show a stronger spatial association between industrial activity and border-serving freight infrastructure than comparable non-border industrial areas in San Diego?

This notebook follows the coding style used in the earlier GPEC 447 homework notebooks:

- short narrative before code blocks;
- clear section headers;
- explicit file paths and required data;
- reusable helper functions;
- CRS checks before distance/buffer operations;
- simple static maps first, optional interactive/extension work later;
- visible checks after each major object is created.

The workflow reflects the project meeting feedback: first make the Otay Mesa case operational, then add comparison areas only after the base workflow runs.

## 0. Required files and data sources

Place local files in `data/raw/` when using downloaded files. FreightViewer layers can also be read directly from URL.

### Minimum required files

1. City of San Diego Community Planning District Boundaries  
2. City of San Diego Zoning  
3. City of San Diego General Plan Land Use  
4. Census tracts or block groups for San Diego County  
5. GeoEnrichment or Business Summary CSV at the same geography as the census units  

### Recommended regional files

6. SANDAG Employment Centers  
7. SANDAG FreightViewer layers:
   - Major Roads
   - POE points
   - POE boundaries
   - truck inspection points or polygons
   - truck parking points or polygons
   - Freight Railroads, if relevant

### Practical principle

Run the Otay Mesa base workflow first. Only add comparison areas after the Otay Mesa maps, indicators, and summary table work.

Cell 1 — Setup

In [1]:
%matplotlib inline

from pathlib import Path
import re
import requests
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)

PROJECT_DIR = Path.cwd()
DATA_PROCESSED = PROJECT_DIR / "data_processed"
FIGURES = PROJECT_DIR / "figures"
TABLES = PROJECT_DIR / "tables"

for folder in [DATA_PROCESSED, FIGURES, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

TARGET_CRS = "EPSG:2230"

print("Project directory:", PROJECT_DIR)
print("Target CRS:", TARGET_CRS)

Project directory: /home/cal081/private/Final Project
Target CRS: EPSG:2230


Cell 2 — Helper functions

In [6]:
def inspect_gdf(gdf, name="GeoDataFrame", n=3):
    """
    Print basic information about a GeoDataFrame.
    Use this after reading every spatial layer.
    """
    print(f"--- {name} ---")
    print("shape:", gdf.shape)
    print("crs:", gdf.crs)
    print("geometry types:")
    print(gdf.geom_type.value_counts(dropna=False))
    print("columns:")
    print(list(gdf.columns))
    display(gdf.head(n))


def to_target_crs(gdf, target_crs=TARGET_CRS, assumed_crs=None):
    """
    Reproject a GeoDataFrame to the target CRS.

    If the source CRS is missing, provide assumed_crs.
    Most online GeoJSON layers are EPSG:4326, but this should still be checked
    after loading.
    """
    if gdf.crs is None:
        if assumed_crs is None:
            raise ValueError("GeoDataFrame has no CRS. Provide assumed_crs.")
        gdf = gdf.set_crs(assumed_crs)

    return gdf.to_crs(target_crs)


def read_online_geojson(url, name, assumed_crs="EPSG:4326"):
    """
    Read a direct online GeoJSON URL and project it to TARGET_CRS.

    This avoids scraping the City of San Diego dataset pages, which can fail
    because the download URLs are not always exposed in easy-to-scrape HTML.
    """
    print(f"Reading {name}")
    print("URL:", url)

    gdf = gpd.read_file(url)

    if gdf.crs is None:
        gdf = gdf.set_crs(assumed_crs)

    gdf = gdf.to_crs(TARGET_CRS)

    inspect_gdf(gdf, name)
    return gdf


def save_optional(gdf, filename):
    """
    Optional cache.

    The notebook is designed to read data online, but saving processed outputs
    helps debugging and avoids repeated web requests during later work.
    """
    out = DATA_PROCESSED / filename
    gdf.to_file(out, driver="GeoJSON")
    print("Saved:", out)


def arcgis_query_url(service_url, layer=0, where="1=1", out_fields="*", out_sr=4326):
    """
    Build a GeoJSON query URL for ArcGIS REST FeatureServer / MapServer layers.

    Example:
    url = arcgis_query_url(
        "https://geo.sandag.org/server/rest/services/Hosted/Roads_Major_SG/FeatureServer",
        layer=0
    )
    gdf = gpd.read_file(url)
    """
    return (
        f"{service_url.rstrip('/')}/{layer}/query?"
        f"where={where}&outFields={out_fields}&outSR={out_sr}&f=geojson"
    )

Cell 3 — Read City of San Diego layers online

In [7]:
city_geojson_urls = {
    "zoning": "https://geo.sandag.org/server/rest/directories/downloads/Zoning_Base_SD.geojson",
    "general_plan": "https://geo.sandag.org/server/rest/directories/downloads/General_Plan_Land_Use_SD.geojson",
    "planning": "https://geo.sandag.org/server/rest/directories/downloads/Community_Plan_SD.geojson",
}

zoning = read_online_geojson(
    city_geojson_urls["zoning"],
    "City of San Diego Zoning"
)

general_plan = read_online_geojson(
    city_geojson_urls["general_plan"],
    "City of San Diego General Plan Land Use"
)

planning = read_online_geojson(
    city_geojson_urls["planning"],
    "City of San Diego Community Planning Districts"
)

Reading City of San Diego Zoning
URL: https://geo.sandag.org/server/rest/directories/downloads/Zoning_Base_SD.geojson
--- City of San Diego Zoning ---
shape: (3706, 7)
crs: EPSG:2230
geometry types:
Polygon         3705
MultiPolygon       1
Name: count, dtype: int64
columns:
['OBJECTID', 'ZONE_NAME', 'IMP_DATE', 'ORDNUM', 'Shape_Length', 'Shape_Area', 'geometry']


,OBJECTID,ZONE_NAME,IMP_DATE,ORDNUM,Shape_Length,Shape_Area,geometry
0,1,CC-3-10,1771545600000,O-22045,4715.752081,537107.479954,"POLYGON ((6314190.843 1860854.877, 6314189.84 ..."
1,2,CC-3-9,1771545600000,O-22045,7805.054966,647065.049254,"POLYGON ((6309944.214 1857677.765, 6309923.557..."
2,3,CC-3-9,1771545600000,O-22045,9537.977279,918383.116510,"POLYGON ((6319110.289 1860715.584, 6319110.111..."


Reading City of San Diego General Plan Land Use
URL: https://geo.sandag.org/server/rest/directories/downloads/General_Plan_Land_Use_SD.geojson
--- City of San Diego General Plan Land Use ---
shape: (62185, 14)
crs: EPSG:2230
geometry types:
Polygon         61926
MultiPolygon      259
Name: count, dtype: int64
columns:
['OBJECTID', 'plan_desc', 'plan_name', 'area_name', 'Density_Low', 'Density_Hi', 'Density_Bonus', 'GP_LU_DESC', 'Color', 'Change_Type', 'Change_Date', 'Shape_Length', 'Shape_Area', 'geometry']


,OBJECTID,plan_desc,plan_name,area_name,Density_Low,Density_Hi,Density_Bonus,GP_LU_DESC,Color,Change_Type,Change_Date,Shape_Length,Shape_Area,geometry
0,1,Residential Low-2,CLAIREMONT,CLAIREMONT,5.0,9.0,NaN,Residential,None,4,12162025.0,329.173850,6384.351778,"POLYGON ((6273018.945 1879707.032, 6272930.626..."
1,2,Very Low Residential,MIRA MESA,Mira Mesa,0.0,4.0,NaN,Residential,None,None,NaN,300.005406,5000.145169,"POLYGON ((6273992.83 1914705.247, 6273949.456 ..."
2,3,Residential Low-2,CLAIREMONT,CLAIREMONT,5.0,9.0,NaN,Residential,None,4,12162025.0,309.358594,5742.033517,"POLYGON ((6266042.957 1880009.978, 6266037 188..."


Reading City of San Diego Community Planning Districts
URL: https://geo.sandag.org/server/rest/directories/downloads/Community_Plan_SD.geojson
--- City of San Diego Community Planning Districts ---
shape: (61, 8)
crs: EPSG:2230
geometry types:
Polygon         60
MultiPolygon     1
Name: count, dtype: int64
columns:
['OBJECTID', 'CPCODE', 'CPNAME', 'ACREAGE', 'Website', 'Shape_Length', 'Shape_Area', 'geometry']


,OBJECTID,CPCODE,CPNAME,ACREAGE,Website,Shape_Length,Shape_Area,geometry
0,1,97,MILITARY FACILITIES,22605.531046,None,182376.005367,9.846930e+08,"POLYGON ((6329243.994 1898826.779, 6329819.554..."
1,2,7,EAST ELLIOTT,2806.769212,https://www.sandiego.gov/planning/community/pr...,68776.912157,1.222624e+08,"MULTIPOLYGON (((6329643.084 1886792.649, 63296..."
2,3,40,TORREY PINES,2722.328674,https://www.sandiego.gov/planning/community/pr...,101824.664393,1.185842e+08,"POLYGON ((6256279.014 1930438.029, 6256267.994..."


Cell 4 — Read SANDAG FreightViewer layers online

In [8]:
freight_urls = {
    "major_roads": "https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json",
    "poe_points": "https://gis.sandag.org/FreightViewer/data/poe_points_update.json",
    "poe_boundaries": "https://gis.sandag.org/FreightViewer/data/poe_boundaries_update.json",
    "truck_inspection_points": "https://gis.sandag.org/FreightViewer/data/truck_inspection_points.json",
    "truck_parking_points": "https://gis.sandag.org/FreightViewer/data/truck_parking_points_20180615.json",
    "freight_rail": "https://gis.sandag.org/FreightViewer/data/Railroads_Freight.json",
}

freight_layers = {}

for key, url in freight_urls.items():
    try:
        gdf = gpd.read_file(url)
        if gdf.crs is None:
            gdf = gdf.set_crs("EPSG:4326")
        gdf = gdf.to_crs(TARGET_CRS)
        freight_layers[key] = gdf
        print(f"{key}: {gdf.shape[0]} features")
    except Exception as e:
        print(f"Failed to load {key}: {e}")

for key in freight_layers:
    inspect_gdf(freight_layers[key], key, n=2)

major_roads: 45 features
poe_points: 10 features
poe_boundaries: 10 features
truck_inspection_points: 2 features
truck_parking_points: 3 features
freight_rail: 2137 features
--- major_roads ---
shape: (45, 11)
crs: EPSG:2230
geometry types:
LineString         29
MultiLineString    16
Name: count, dtype: int64
columns:
['OBJECTID_1', 'OBJECTID', 'IFC', 'LABEL', 'EXISTING', 'Shape_Leng', 'Shape_Le_1', 'NM', 'Source', 'Mileage', 'geometry']


,OBJECTID_1,OBJECTID,IFC,LABEL,EXISTING,Shape_Leng,Shape_Le_1,NM,Source,Mileage,geometry
0,5,1,1,15,1,109.350581,577371.069707,Interstate 15,SANDAG,109.3,"MULTILINESTRING ((6294369.903 1832252.359, 629..."
1,6,2,2,76,1,51.965596,274378.347456,State Route 76,SANDAG,52.0,"MULTILINESTRING ((6420868.803 2016622.487, 642..."


--- poe_points ---
shape: (10, 10)
crs: EPSG:2230
geometry types:
Point    10
Name: count, dtype: int64
columns:
['OBJECTID_1', 'Status', 'Existing', 'Future', 'Port_ID', 'port_name', 'url', 'wait_time', 'mode', 'geometry']


,OBJECTID_1,Status,Existing,Future,Port_ID,port_name,url,wait_time,mode,geometry
0,1,Existing,1,0,250602,Otay Mesa Commercial,https://www.cbp.gov/contact/ports/otay-mesa,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,Commercial Vehicle,POINT (6350761.598 1781024.687)
1,2,Existing,1,0,250201,Andrade,https://www.cbp.gov/contact/ports/andrade-class,https://bwt.cbp.gov/index.html?com=1&pas=1&ped...,"Passenger Vehicle, Pedestrian",POINT (7029714.359 1844608.768)


--- poe_boundaries ---
shape: (10, 13)
crs: EPSG:2230
geometry types:
Polygon         9
MultiPolygon    1
Name: count, dtype: int64
columns:
['OBJECTID', 'NAME', 'Status', 'Port_ID', 'Existing', 'Future', 'port_name', 'url', 'wait_time', 'mode', 'Shape_Leng', 'Shape_Area', 'geometry']


...

,OBJECTID,Name,Source,Operator,Volume,Lanes,Acres,url,geometry
0,1,San Onofre Commercial Vehicle Enforcement Faci...,SANDAG,California Highway Patrol,"180,000 commercial vehicles per month",0,0,<a href='https://www.chp.ca.gov/find-an-office...,POINT (6172278.195 2074217.724)
1,2,Otay Mesa Commercial Vehicle Enforcement Facility,SANDAG,California Highway Patrol Border Division,None,0,0,<a href='https://www.chp.ca.gov/find-an-office...,POINT (6353098.889 1782310.805)


--- truck_parking_points ---
shape: (3, 10)
crs: EPSG:2230
geometry types:
Point    3
Name: count, dtype: int64
columns:
['OBJECTID', 'PARKING_ID', 'NAME', 'Source', 'ACTIVE', 'STATUS', 'ORIG_FID', 'Amenities', 'Spaces', 'geometry']


,OBJECTID,PARKING_ID,NAME,Source,ACTIVE,STATUS,ORIG_FID,Amenities,Spaces,geometry
0,1,1,Aliso Creek Rest Area,SANDAG,Yes,Rest Area,1,"Restrooms, Water, Picnic Tables, Phone, Handic...",27,POINT (6196772.012 2044906.759)
1,3,2,Buckman Springs Rest Area,SANDAG,Yes,Rest Area,2,"Restrooms, Water, Picnic Tables, Phone, Handic...",18,POINT (6489767.727 1855639.082)


--- freight_rail ---
shape: (2137, 25)
crs: EPSG:2230
geometry types:
LineString    2137
Name: count, dtype: int64
columns:
['Shape_STLe', 'd_SvcType2', 'd_SvcType1', 'SvcName1', 'SvcName3', 'SvcName2', 'd_SvcType3', 'd_Owner', 'InSD', 'd_SvcName1', 'd_SvcName2', 'd_SvcName3', 'TrackType', 'd_Operator', 'Source', 'SvcType2', 'SvcType3', 'd_Operat_1', 'SvcType1', 'd_Operat_2', 'Operator1', 'Operator2', 'd_TrackTyp', 'Owner', 'geometry']


,Shape_STLe,d_SvcType2,d_SvcType1,SvcName1,SvcName3,SvcName2,d_SvcType3,d_Owner,InSD,d_SvcName1,d_SvcName2,d_SvcName3,TrackType,d_Operator,Source,SvcType2,SvcType3,d_Operat_1,SvcType1,d_Operat_2,Operator1,Operator2,d_TrackTyp,Owner,geometry
0,15874.079779,Intercity Rail,Commuter Rail,3,8,4,Freight Rail,NCTD,1,Metrolink,Pacific Surfliner,Freight,1,SCRRA,SANDAG,2,4,Amtrak,1,BNSF,2,Amtrak,Mainline,1,"LINESTRING (6159193.302 2083871.462, 6159389.4..."
1,4787.511707,Intercity Rail,Commuter Rail,3,8,4,Freight Rail,NCTD,1,Metrolink,Pacific Surfliner,Freight,1,SCRRA,SANDAG,2,4,Amtrak,1,BNSF,2,Amtrak,Mainline,1,"LINESTRING (6193090.582 2051768.96, 6193164.97..."


Cell 5 — Read Census tracts online

In [9]:
tract_url = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_06_tract_500k.zip"

tracts_ca = gpd.read_file(tract_url)
tracts_sd = tracts_ca[tracts_ca["COUNTYFP"] == "073"].copy()
tracts_sd = tracts_sd.to_crs(TARGET_CRS)
tracts_sd["area_sqmi"] = tracts_sd.geometry.area / (5280 ** 2)

inspect_gdf(tracts_sd, "San Diego County Census Tracts")

--- San Diego County Census Tracts ---
shape: (736, 15)
crs: EPSG:2230
geometry types:
Polygon         735
MultiPolygon      1
Name: count, dtype: int64
columns:
['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOIDFQ', 'GEOID', 'NAME', 'NAMELSAD', 'STUSPS', 'NAMELSADCO', 'STATE_NAME', 'LSAD', 'ALAND', 'AWATER', 'geometry', 'area_sqmi']


,STATEFP,COUNTYFP,TRACTCE,GEOIDFQ,GEOID,NAME,NAMELSAD,STUSPS,NAMELSADCO,STATE_NAME,LSAD,ALAND,AWATER,geometry,area_sqmi
131,06,073,003212,1400000US06073003212,06073003212,32.12,Census Tract 32.12,CA,San Diego County,California,CT,1257736,0,"POLYGON ((6311880.924 1822303.944, 6312259.685...",0.491883
132,06,073,013104,1400000US06073013104,06073013104,131.04,Census Tract 131.04,CA,San Diego County,California,CT,977893,0,"POLYGON ((6304832.054 1805358.979, 6306101.849...",0.377696
133,06,073,020902,1400000US06073020902,06073020902,209.02,Census Tract 209.02,CA,San Diego County,California,CT,422787620,7987272,"POLYGON ((6395854.291 1910128.055, 6396039.583...",166.233949


Cell 6 — Pull ACS industry variables online

In [11]:
# Cell 6 — Pull ACS industry variables online

acs_year = 2023

# ACS 5-year Data Profile variables
# DP03_0033E: Civilian employed population 16 years and over
# DP03_0034E: Construction
# DP03_0035E: Manufacturing
# DP03_0036E: Wholesale trade
# DP03_0038E: Transportation and warehousing, and utilities
acs_vars = [
    "NAME",
    "DP03_0033E",
    "DP03_0034E",
    "DP03_0035E",
    "DP03_0036E",
    "DP03_0038E",
]

# IMPORTANT: the correct endpoint is /acs/acs5/profile
base = f"https://api.census.gov/data/{acs_year}/acs/acs5/profile"

params = {
    "get": ",".join(acs_vars),
    "for": "tract:*",
    "in": "state:06 county:073"
}

resp = requests.get(base, params=params, timeout=60)

print("Request URL:")
print(resp.url)
print("Status code:", resp.status_code)
print("Response preview:")
print(resp.text[:500])

# Stop early if the request failed
resp.raise_for_status()

# Parse JSON only after checking the response
try:
    data = resp.json()
except Exception as e:
    raise ValueError(
        "The Census API response was not valid JSON. "
        "Check the request URL, endpoint, variables, and Census API availability. "
        f"Response preview: {resp.text[:500]}"
    ) from e

acs = pd.DataFrame(data[1:], columns=data[0])

# Create GEOID
acs["GEOID"] = acs["state"] + acs["county"] + acs["tract"]

rename = {
    "DP03_0033E": "employed_total",
    "DP03_0034E": "emp_construction",
    "DP03_0035E": "emp_manufacturing",
    "DP03_0036E": "emp_wholesale",
    "DP03_0038E": "emp_transport_warehousing_utilities",
}

acs = acs.rename(columns=rename)

num_cols = list(rename.values())

for c in num_cols:
    acs[c] = pd.to_numeric(acs[c], errors="coerce")

# Public-data industrial activity proxy
acs["industrial_proxy_employment"] = (
    acs["emp_construction"].fillna(0)
    + acs["emp_manufacturing"].fillna(0)
    + acs["emp_wholesale"].fillna(0)
    + acs["emp_transport_warehousing_utilities"].fillna(0)
)

acs["industrial_proxy_share"] = (
    acs["industrial_proxy_employment"] / acs["employed_total"]
)

display(acs.head())
print("ACS rows and columns:", acs.shape)

Request URL:
https://api.census.gov/data/missing_key.html
Status code: 200
Response preview:

<html>
    <head>
        <title>Missing Key</title>
    </head>
    <body>
        <p>
            A valid <em>key</em> must be included with each data API request.
            If you do not have a key, you may sign up for one <a href="key_signup.html">here</a>.
        </p>
        <p>
        	If you have questions, please reach out to census.data@census.gov.
        </p>
    </body>
</html>



ValueError: The Census API response was not valid JSON. Check the request URL, endpoint, variables, and Census API availability. Response preview: 
<html>
    <head>
        <title>Missing Key</title>
    </head>
    <body>
        <p>
            A valid <em>key</em> must be included with each data API request.
            If you do not have a key, you may sign up for one <a href="key_signup.html">here</a>.
        </p>
        <p>
        	If you have questions, please reach out to census.data@census.gov.
        </p>
    </body>
</html>


Cell 7 — Join ACS to tracts

In [ ]:
tracts_industry = tracts_sd.merge(
    acs[
        [
            "GEOID",
            "NAME",
            "employed_total",
            "emp_construction",
            "emp_manufacturing",
            "emp_wholesale",
            "emp_transport_warehousing_utilities",
            "industrial_proxy_employment",
            "industrial_proxy_share",
        ]
    ],
    on="GEOID",
    how="left"
)

tracts_industry["industrial_proxy_density"] = (
    tracts_industry["industrial_proxy_employment"] / tracts_industry["area_sqmi"]
)

inspect_gdf(tracts_industry, "San Diego tracts with ACS industrial proxy")

Cell 8 — Define Otay Mesa study area

In [ ]:
print("Planning columns:", list(planning.columns))

possible_name_cols = [
    c for c in planning.columns
    if any(token in c.lower() for token in ["name", "community", "plan"])
]
print("Possible name columns:", possible_name_cols)

planning_name_col = possible_name_cols[0]
planning["name_check"] = planning[planning_name_col].astype(str).str.lower()

otay_candidates = planning[planning["name_check"].str.contains("otay", na=False)].copy()
inspect_gdf(otay_candidates, "Otay-related planning areas")

ax = planning.plot(figsize=(10, 8), facecolor="none", edgecolor="lightgray")
otay_candidates.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
plt.title("Otay-related Planning Area Candidates")
plt.axis("off")
plt.show()

In [ ]:
display(otay_candidates[[planning_name_col, "geometry"]].drop(columns="geometry"))

otay_mesa = otay_candidates.copy()

otay_context = otay_mesa.copy()
otay_context["geometry"] = otay_context.geometry.buffer(5 * 5280)

inspect_gdf(otay_mesa, "Selected Otay Mesa study area")

Cell 9 — Clip tracts and freight layers

In [ ]:
tracts_otay = gpd.clip(tracts_industry, otay_context)

freight_otay = {}
for key, gdf in freight_layers.items():
    try:
        freight_otay[key] = gpd.clip(gdf, otay_context)
        print(key, freight_otay[key].shape)
    except Exception as e:
        print(f"Could not clip {key}: {e}")

inspect_gdf(tracts_otay, "Otay Mesa context tracts")

Cell 10 — First diagnostic map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

otay_context.boundary.plot(ax=ax, color="black", linewidth=1)
otay_mesa.boundary.plot(ax=ax, color="red", linewidth=2)

if "major_roads" in freight_otay:
    freight_otay["major_roads"].plot(ax=ax, color="gray", linewidth=1)

if "freight_rail" in freight_otay:
    freight_otay["freight_rail"].plot(ax=ax, color="purple", linewidth=1, alpha=0.7)

if "poe_points" in freight_otay:
    freight_otay["poe_points"].plot(ax=ax, color="red", markersize=70)

if "truck_parking_points" in freight_otay:
    freight_otay["truck_parking_points"].plot(ax=ax, color="blue", markersize=20, alpha=0.8)

if "truck_inspection_points" in freight_otay:
    freight_otay["truck_inspection_points"].plot(ax=ax, color="orange", markersize=20, alpha=0.8)

plt.title("Otay Mesa Context and Freight Infrastructure")
plt.axis("off")
plt.tight_layout()
plt.show()

## 1. Setup: packages, folders, and CRS

This notebook uses the same environment style as the homework notebooks: imports are grouped near the top, project folders are created explicitly, and San Diego distance/area operations use a projected CRS.

The earlier homework used `os.environ["HOME"] + "/public/datasets/"` for course datasets. This project also creates local project folders so that downloaded public data and outputs are kept together.

In [ ]:
%matplotlib inline

import os
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import shapely
from shapely.geometry import Point

pd.set_option("display.max_columns", 120)

# Course-style shared data location; useful if running on JupyterHub.
data_location = os.environ.get("HOME", "") + "/public/datasets/"

# Project folders.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"
FIGURES = PROJECT_DIR / "figures"
TABLES = PROJECT_DIR / "tables"

for folder in [DATA_RAW, DATA_PROCESSED, FIGURES, TABLES]:
    folder.mkdir(parents=True, exist_ok=True)

# EPSG:2230 is appropriate for local San Diego distance and area work in feet.
TARGET_CRS = "EPSG:2230"

print("Project directory:", PROJECT_DIR)
print("Shared data location:", data_location)
print("Raw data folder:", DATA_RAW)
print("Target CRS:", TARGET_CRS)

## 2. Helper functions

These helpers mirror the homework pattern of checking objects after creating them. Use them throughout the notebook to avoid silent CRS, geometry, or column mistakes.

In [ ]:
def project_check(var_name):
    '''
    Lightweight self-check helper, modeled after the HW1 checking style.
    '''
    if var_name not in globals():
        print(f"{var_name}: not defined yet")
        return

    obj = globals()[var_name]
    print(f"{var_name}: defined")
    print("type:", type(obj))

    if hasattr(obj, "shape"):
        print("shape:", obj.shape)
    if hasattr(obj, "crs"):
        print("crs:", obj.crs)
    if hasattr(obj, "geom_type"):
        print("geometry types:")
        print(obj.geom_type.value_counts(dropna=False))


def inspect_gdf(gdf, name="GeoDataFrame", n=5):
    '''
    Print common GeoDataFrame diagnostics.
    '''
    print(f"--- {name} ---")
    print("shape:", gdf.shape)
    print("crs:", gdf.crs)
    print("geometry types:")
    print(gdf.geom_type.value_counts(dropna=False))
    print("columns:")
    print(list(gdf.columns))
    display(gdf.head(n))


def safe_to_crs(gdf, target_crs=TARGET_CRS, assumed_crs=None):
    '''
    Reproject a GeoDataFrame. If CRS is missing, optionally set an assumed CRS first.
    '''
    if gdf.crs is None:
        if assumed_crs is None:
            raise ValueError("GeoDataFrame has no CRS. Provide assumed_crs before projecting.")
        gdf = gdf.set_crs(assumed_crs)
    return gdf.to_crs(target_crs)


def save_gdf(gdf, filename):
    '''
    Save a GeoDataFrame to data/processed as GeoJSON.
    '''
    out = DATA_PROCESSED / filename
    gdf.to_file(out, driver="GeoJSON")
    print("Saved:", out)


def read_local_or_url(local_path, url=None, assumed_crs=None, target_crs=TARGET_CRS):
    '''
    Read a local geospatial file if it exists. Otherwise try the URL.
    '''
    local_path = Path(local_path)

    if local_path.exists():
        gdf = gpd.read_file(local_path)
        print("Loaded local file:", local_path)
    elif url is not None:
        gdf = gpd.read_file(url)
        print("Loaded URL:", url)
    else:
        raise FileNotFoundError(f"No local file found and no URL supplied: {local_path}")

    return safe_to_crs(gdf, target_crs=target_crs, assumed_crs=assumed_crs)


def quick_map(gdf, title=None, column=None, figsize=(10, 8), **kwargs):
    '''
    Quick GeoPandas map with a title and no axes.
    '''
    ax = gdf.plot(column=column, figsize=figsize, legend=column is not None, **kwargs)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()
    return ax

## 3. Load City of San Diego planning and land-use layers


Update the local file names after downloading data. The code is written so that the notebook does not assume exact file names or columns until you inspect them.

**Expected use:**
1. Download each layer into `data/raw/`.
2. Update the path variables below.
3. Run the cells and inspect the columns.

In [ ]:
# -------------------------------------------------------------------
# Update these file names after downloading the layers.
# GeoJSON is easiest, but shapefiles also work.
# -------------------------------------------------------------------

zoning_path = DATA_RAW / "zoning.geojson"
general_plan_path = DATA_RAW / "general_plan_land_use.geojson"
planning_path = DATA_RAW / "community_planning_districts.geojson"
employment_centers_path = DATA_RAW / "sandag_employment_centers.geojson"

# Uncomment when files are available.
# zoning = read_local_or_url(zoning_path)
# general_plan = read_local_or_url(general_plan_path)
# planning = read_local_or_url(planning_path)
# employment_centers = read_local_or_url(employment_centers_path)

# project_check("zoning")
# project_check("general_plan")
# project_check("planning")
# project_check("employment_centers")

In [ ]:
# Inspect columns after loading.
# inspect_gdf(zoning, "City of San Diego Zoning")
# inspect_gdf(general_plan, "General Plan Land Use")
# inspect_gdf(planning, "Community Planning Districts")
# inspect_gdf(employment_centers, "SANDAG Employment Centers")

## 4. Define Otay Mesa study area


The meeting feedback was to first make Otay Mesa the base case. This section extracts Otay Mesa from the planning boundary layer and creates a local context buffer for maps and nearby freight infrastructure.

If the planning layer uses a different name field, update `planning_name_col`.

In [ ]:
# -------------------------------------------------------------------
# Run after loading planning.
# -------------------------------------------------------------------

# planning_name_col = "name"  # update after inspecting planning.columns
# planning["name_check"] = planning[planning_name_col].astype(str).str.lower()

# otay_mesa = planning[
#     planning["name_check"].str.contains("otay mesa", na=False)
# ].copy()

# inspect_gdf(otay_mesa, "Otay Mesa boundary candidates")

In [ ]:
# Plot Otay Mesa candidate(s) to verify.

# ax = planning.plot(figsize=(10, 8), facecolor="none", edgecolor="lightgray")
# otay_mesa.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
# plt.title("Otay Mesa Study Area Candidate")
# plt.axis("off")
# plt.show()

# Create a 5-mile context area around Otay Mesa.
# otay_context = otay_mesa.copy()
# otay_context["geometry"] = otay_context.geometry.buffer(5 * 5280)

# save_gdf(otay_mesa, "otay_mesa_boundary.geojson")
# save_gdf(otay_context, "otay_mesa_5mile_context.geojson")

## 5. Load SANDAG FreightViewer layers

These are the instructor-recommended FreightViewer layers. The code reads them directly from URL and saves local copies for reproducibility.

The core analytical layers are:
- ports of entry;
- major roads / freight corridors;
- truck inspection;
- truck parking;
- freight rail, if spatially relevant.

In [ ]:
freight_urls = {
    "major_roads": "https://gis.sandag.org/FreightViewer/data/MajorRoads_20171002.json",
    "poe_points": "https://gis.sandag.org/FreightViewer/data/poe_points_update.json",
    "poe_boundaries": "https://gis.sandag.org/FreightViewer/data/poe_boundaries_update.json",
    "truck_inspection_points": "https://gis.sandag.org/FreightViewer/data/truck_inspection_points.json",
    "truck_inspection_polys": "https://gis.sandag.org/FreightViewer/data/truck_inspection_polys.json",
    "truck_parking_points": "https://gis.sandag.org/FreightViewer/data/truck_parking_points_20180615.json",
    "truck_parking_polys": "https://gis.sandag.org/FreightViewer/data/truck_parking_polys_20180615.json",
    "freight_rail": "https://gis.sandag.org/FreightViewer/data/Railroads_Freight.json",
    "rail_yards_points": "https://gis.sandag.org/FreightViewer/data/Rail_Yards_Points.json",
    "rail_yards_polys": "https://gis.sandag.org/FreightViewer/data/rail_yards_poly_dom.json",
}

freight_layers = {}

for key, url in freight_urls.items():
    try:
        gdf = gpd.read_file(url)
        # Most URL layers are in lon/lat. If CRS metadata exists, GeoPandas will use it.
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326)
        gdf = gdf.to_crs(TARGET_CRS)
        freight_layers[key] = gdf
        print(f"{key}: loaded {gdf.shape[0]} features")
    except Exception as e:
        print(f"{key}: could not load")
        print("  ", e)

In [ ]:
# Inspect selected core layers.
for key in ["major_roads", "poe_points", "truck_inspection_points", "truck_parking_points", "freight_rail"]:
    if key in freight_layers:
        inspect_gdf(freight_layers[key], key, n=3)

# Save local copies.
for key, gdf in freight_layers.items():
    try:
        save_gdf(gdf, f"{key}.geojson")
    except Exception as e:
        print(f"Could not save {key}: {e}")

## 6. Clip layers to Otay Mesa context

This follows the homework style of subsetting before analysis. It makes maps clearer and keeps later spatial operations faster.

Run after `otay_context` has been created.

In [ ]:
# -------------------------------------------------------------------
# Run after otay_context exists.
# -------------------------------------------------------------------

# zoning_otay = gpd.clip(zoning, otay_context)
# general_plan_otay = gpd.clip(general_plan, otay_context)
# employment_otay = gpd.clip(employment_centers, otay_context)

# freight_otay = {}
# for key, gdf in freight_layers.items():
#     try:
#         freight_otay[key] = gpd.clip(gdf, otay_context)
#         print(key, len(freight_otay[key]))
#     except Exception as e:
#         print(f"Could not clip {key}: {e}")

# project_check("zoning_otay")
# project_check("general_plan_otay")
# project_check("employment_otay")

## 7. Map 1: Otay Mesa and freight infrastructure

This first map is a diagnostic map. It confirms that the study area, POEs, roads, and freight facilities line up correctly.

This is similar to the layered-map approach used in the homework: plot a base polygon, then add points/lines on top.

In [ ]:
# -------------------------------------------------------------------
# Run after freight_otay and otay_mesa exist.
# -------------------------------------------------------------------

# fig, ax = plt.subplots(figsize=(10, 8))

# otay_context.boundary.plot(ax=ax, color="black", linewidth=1)
# otay_mesa.boundary.plot(ax=ax, color="red", linewidth=2)

# if "major_roads" in freight_otay:
#     freight_otay["major_roads"].plot(ax=ax, color="gray", linewidth=1)

# if "freight_rail" in freight_otay:
#     freight_otay["freight_rail"].plot(ax=ax, color="purple", linewidth=1, alpha=0.7)

# if "poe_points" in freight_otay:
#     freight_otay["poe_points"].plot(ax=ax, color="red", markersize=60)

# if "truck_parking_points" in freight_otay:
#     freight_otay["truck_parking_points"].plot(ax=ax, color="blue", markersize=20, alpha=0.8)

# if "truck_inspection_points" in freight_otay:
#     freight_otay["truck_inspection_points"].plot(ax=ax, color="orange", markersize=20, alpha=0.8)

# plt.title("Otay Mesa Study Area and Freight Infrastructure")
# plt.axis("off")
# plt.tight_layout()
# plt.savefig(FIGURES / "map1_otay_mesa_freight_infrastructure.png", dpi=300)
# plt.show()

## 8. Define industrially relevant land

This part keeps the land-use definition transparent. It should be used for mapping and for checking how zoning/planning relates to actual industrial activity.

The main analysis should still use census units plus GeoEnrichment variables if those data are available.

In [ ]:
# -------------------------------------------------------------------
# Update field names after inspecting zoning and general_plan.
# -------------------------------------------------------------------

# zoning_code_col = "zone_code"
# gp_landuse_col = "land_use"

industrial_prefixes = ("IP", "IL", "IH", "IS", "IBT")
industrial_landuse_keywords = ["industrial", "employment"]

# Example zoning filter.
# zoning_industrial = zoning[
#     zoning[zoning_code_col].astype(str).str.startswith(industrial_prefixes, na=False)
# ].copy()

# Example General Plan filter.
# general_plan_industrial = general_plan[
#     general_plan[gp_landuse_col].astype(str).str.lower().str.contains(
#         "|".join(industrial_landuse_keywords),
#         na=False
#     )
# ].copy()

# inspect_gdf(zoning_industrial, "Industrial zoning")
# inspect_gdf(general_plan_industrial, "Industrial / employment-oriented General Plan land use")

In [ ]:
# Strict definition: both zoning and General Plan indicate industrial/employment relevance.

# strict_industrial = gpd.overlay(
#     zoning_industrial,
#     general_plan_industrial,
#     how="intersection"
# )

# Broad definition: either zoning or General Plan indicates industrial/employment relevance.

# zoning_simple = zoning_industrial[["geometry"]].copy()
# zoning_simple["source"] = "zoning"

# gp_simple = general_plan_industrial[["geometry"]].copy()
# gp_simple["source"] = "general_plan"

# broad_industrial = pd.concat([zoning_simple, gp_simple], ignore_index=True)
# broad_industrial = gpd.GeoDataFrame(broad_industrial, geometry="geometry", crs=TARGET_CRS)

# broad_industrial["group"] = 1
# broad_industrial_dissolved = broad_industrial.dissolve(by="group").reset_index()

# save_gdf(strict_industrial, "strict_industrial_land.geojson")
# save_gdf(broad_industrial_dissolved, "broad_industrial_land.geojson")

## 9. Prepare census tracts or block groups

This is the recommended main unit of analysis after the meeting. Census units give more observations than a few industrial zones and can be enriched with business variables.

The cell below uses `pygris` if available. If not, download Census boundaries manually and read them as a local file.

In [ ]:
# Option A: use pygris. Uncomment if pygris is installed.
# !pip install pygris

# import pygris
# tracts = pygris.tracts(state="CA", county="San Diego", cb=True, year=2020)
# tracts = tracts.to_crs(TARGET_CRS)

# inspect_gdf(tracts, "San Diego County Census Tracts")

In [ ]:
# Option B: read a local tract file.
# tracts_path = DATA_RAW / "CENSUS_TRACT_TIGER2020.shp"
# tracts = gpd.read_file(tracts_path)
# tracts = safe_to_crs(tracts, TARGET_CRS)

# inspect_gdf(tracts, "San Diego County Census Tracts")

# Clip census units to the Otay Mesa context.
# tracts_otay = gpd.clip(tracts, otay_context)
# tracts_otay["area_sqmi"] = tracts_otay.geometry.area / (5280 ** 2)

# inspect_gdf(tracts_otay, "Otay Mesa context census units")
# save_gdf(tracts_otay, "tracts_otay_context.geojson")

## 10. Add GeoEnrichment / Business Summary variables

The meeting suggested using Esri GeoEnrichment or Business Summary variables to measure actual industrial activity by census unit.

Export a CSV using the same geography as your census unit layer and save it as:

`data/raw/geoenrichment_industrial_units.csv`

Potential variables:
- industrial employment;
- number of industrial businesses;
- business sales/revenue;
- manufacturing or wholesale business counts;
- production-related occupations.

After loading, update the ID and variable names below.

In [ ]:
geo_csv_path = DATA_RAW / "geoenrichment_industrial_units.csv"

# Uncomment after export is ready.
# industrial_vars = pd.read_csv(geo_csv_path)
# display(industrial_vars.head())
# print(industrial_vars.columns.tolist())

In [ ]:
# -------------------------------------------------------------------
# Update these names after inspecting the CSV.
# -------------------------------------------------------------------

# geoid_col = "GEOID"
# industrial_employment_col = "industrial_employment"
# industrial_businesses_col = "industrial_businesses"
# business_sales_col = "business_sales"

# industrial_vars[geoid_col] = industrial_vars[geoid_col].astype(str)
# tracts_otay["GEOID"] = tracts_otay["GEOID"].astype(str)

# tracts_otay = tracts_otay.merge(
#     industrial_vars,
#     left_on="GEOID",
#     right_on=geoid_col,
#     how="left"
# )

# tracts_otay = tracts_otay.rename(columns={
#     industrial_employment_col: "industrial_employment",
#     industrial_businesses_col: "industrial_businesses",
#     business_sales_col: "business_sales"
# })

# Density measures.
# tracts_otay["industrial_emp_density"] = tracts_otay["industrial_employment"] / tracts_otay["area_sqmi"]
# tracts_otay["industrial_business_density"] = tracts_otay["industrial_businesses"] / tracts_otay["area_sqmi"]

## 11. Distance to ports of entry and freight corridors

This section turns the research question into measurable proximity variables.

For census polygons, the notebook uses representative points for distance measures. This is a practical simplification. Area overlap with buffers is calculated using the full polygon geometry.

In [ ]:
# -------------------------------------------------------------------
# Run after tracts_otay and freight_layers are ready.
# -------------------------------------------------------------------

# tracts_otay["rep_geom"] = tracts_otay.geometry.representative_point()
# tract_points = tracts_otay.set_geometry("rep_geom").copy()

# POE points.
# poe = freight_layers["poe_points"].copy()
# inspect_gdf(poe, "POE points")

# Update name field after inspection.
# poe_name_col = "Name"
# poe["name_lower"] = poe[poe_name_col].astype(str).str.lower()
# poe_core = poe[poe["name_lower"].str.contains("otay|san ysidro", na=False)].copy()

# If name filtering does not work, use clipped POEs:
# poe_core = gpd.clip(poe, otay_context)

# poe_union = poe_core.union_all()
# tracts_otay["dist_to_nearest_poe_ft"] = tract_points.geometry.apply(lambda geom: geom.distance(poe_union))
# tracts_otay["dist_to_nearest_poe_mi"] = tracts_otay["dist_to_nearest_poe_ft"] / 5280

# Major roads / freight corridor distance.
# major_roads = freight_layers["major_roads"].copy()
# major_roads_union = major_roads.union_all()

# tracts_otay["dist_to_major_road_ft"] = tract_points.geometry.apply(lambda geom: geom.distance(major_roads_union))
# tracts_otay["dist_to_major_road_mi"] = tracts_otay["dist_to_major_road_ft"] / 5280

## 12. Freight corridor buffers and area-share overlap

This follows the buffer logic from the GeoPandas homework. Instead of only asking whether a unit intersects a corridor buffer, we calculate how much of each polygon's area lies inside the buffer.

In [ ]:
# -------------------------------------------------------------------
# Run after major_roads and tracts_otay are available.
# -------------------------------------------------------------------

# buffer_specs = {
#     "buffer_0_5mi": 0.5 * 5280,
#     "buffer_1mi": 1 * 5280,
#     "buffer_2mi": 2 * 5280,
# }

# road_buffers = {}

# for name, dist in buffer_specs.items():
#     buffer_union = major_roads.buffer(dist).union_all()
#     road_buffers[name] = gpd.GeoDataFrame(
#         {"buffer_name": [name]},
#         geometry=[buffer_union],
#         crs=TARGET_CRS,
#     )

# tracts_otay["tract_area"] = tracts_otay.geometry.area

# for name, buffer_gdf in road_buffers.items():
#     tracts_otay[f"intersects_{name}"] = tracts_otay.geometry.intersects(buffer_gdf.geometry.iloc[0])

#     overlap = gpd.overlay(
#         tracts_otay[["GEOID", "geometry", "tract_area"]],
#         buffer_gdf,
#         how="intersection",
#     )
#     overlap["overlap_area"] = overlap.geometry.area

#     overlap_summary = overlap.groupby("GEOID", as_index=False)["overlap_area"].sum()
#     overlap_summary = overlap_summary.rename(columns={"overlap_area": f"overlap_area_{name}"})

#     tracts_otay = tracts_otay.merge(overlap_summary, on="GEOID", how="left")
#     tracts_otay[f"overlap_area_{name}"] = tracts_otay[f"overlap_area_{name}"].fillna(0)
#     tracts_otay[f"share_inside_{name}"] = (
#         tracts_otay[f"overlap_area_{name}"] / tracts_otay["tract_area"]
#     )

## 13. Build the main analysis table

This table is the central product for statistics, charts, and final writing.

In [ ]:
# core_cols = [
#     "GEOID",
#     "industrial_employment",
#     "industrial_businesses",
#     "business_sales",
#     "area_sqmi",
#     "industrial_emp_density",
#     "industrial_business_density",
#     "dist_to_nearest_poe_mi",
#     "dist_to_major_road_mi",
#     "share_inside_buffer_0_5mi",
#     "share_inside_buffer_1mi",
#     "share_inside_buffer_2mi",
#     "geometry",
# ]

# existing_cols = [c for c in core_cols if c in tracts_otay.columns]
# analysis_gdf = tracts_otay[existing_cols].copy()

# save_gdf(analysis_gdf, "otay_mesa_analysis_units.geojson")
# analysis_gdf.drop(columns="geometry").to_csv(
#     TABLES / "otay_mesa_industrial_logistics_indicators.csv",
#     index=False
# )

# display(analysis_gdf.head())

## 14. Summary statistics and exploratory plots

These are not causal tests. They are exploratory summaries of spatial association.

In [ ]:
# summary_vars = [
#     "industrial_emp_density",
#     "industrial_business_density",
#     "dist_to_nearest_poe_mi",
#     "dist_to_major_road_mi",
#     "share_inside_buffer_1mi",
# ]

# existing_summary_vars = [c for c in summary_vars if c in analysis_gdf.columns]

# summary_stats = analysis_gdf[existing_summary_vars].describe()
# display(summary_stats)
# summary_stats.to_csv(TABLES / "otay_mesa_summary_statistics.csv")

# Correlation matrix.
# corr = analysis_gdf[existing_summary_vars].corr()
# display(corr)
# corr.to_csv(TABLES / "otay_mesa_correlation_matrix.csv")

In [ ]:
# Scatter plot: industrial employment density vs major road distance.

# plt.figure(figsize=(7, 5))
# plt.scatter(
#     analysis_gdf["dist_to_major_road_mi"],
#     analysis_gdf["industrial_emp_density"],
#     alpha=0.8
# )
# plt.xlabel("Distance to Major Freight Road (miles)")
# plt.ylabel("Industrial Employment Density")
# plt.title("Industrial Employment Density and Freight Road Proximity")
# plt.tight_layout()
# plt.savefig(FIGURES / "industrial_employment_vs_road_distance.png", dpi=300)
# plt.show()

## 15. Final map set

Each map should answer one question:

1. Where is the Otay Mesa study area and freight infrastructure?
2. Where is industrial activity strongest?
3. Which census units overlap freight corridor buffers?
4. Which census units are closest to the POEs?

### Map 1: Study area and freight infrastructure

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 8))
# otay_context.boundary.plot(ax=ax, color="black", linewidth=1)
# otay_mesa.boundary.plot(ax=ax, color="red", linewidth=2)

# freight_otay["major_roads"].plot(ax=ax, color="gray", linewidth=1)

# if "freight_rail" in freight_otay:
#     freight_otay["freight_rail"].plot(ax=ax, color="purple", linewidth=1, alpha=0.7)

# freight_otay["poe_points"].plot(ax=ax, color="red", markersize=60)

# if "truck_parking_points" in freight_otay:
#     freight_otay["truck_parking_points"].plot(ax=ax, color="blue", markersize=20, alpha=0.8)

# if "truck_inspection_points" in freight_otay:
#     freight_otay["truck_inspection_points"].plot(ax=ax, color="orange", markersize=20, alpha=0.8)

# plt.title("Otay Mesa Study Area and Freight Infrastructure")
# plt.axis("off")
# plt.tight_layout()
# plt.savefig(FIGURES / "map1_otay_mesa_freight_infrastructure.png", dpi=300)
# plt.show()

### Map 2: Industrial employment density

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 8))

# analysis_gdf.plot(
#     column="industrial_emp_density",
#     legend=True,
#     ax=ax,
#     edgecolor="white",
#     linewidth=0.3,
#     missing_kwds={"color": "lightgray", "label": "Missing"},
# )

# freight_otay["major_roads"].plot(ax=ax, color="black", linewidth=1)
# freight_otay["poe_points"].plot(ax=ax, color="red", markersize=50)

# plt.title("Industrial Employment Density by Census Unit")
# plt.axis("off")
# plt.tight_layout()
# plt.savefig(FIGURES / "map2_industrial_employment_density.png", dpi=300)
# plt.show()

### Map 3: 1-mile freight corridor buffer overlap

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 8))

# analysis_gdf.plot(
#     column="share_inside_buffer_1mi",
#     legend=True,
#     ax=ax,
#     edgecolor="white",
#     linewidth=0.3,
#     missing_kwds={"color": "lightgray", "label": "Missing"},
# )

# road_buffers["buffer_1mi"].boundary.plot(ax=ax, color="blue", linewidth=1)
# freight_otay["major_roads"].plot(ax=ax, color="black", linewidth=1)
# freight_otay["poe_points"].plot(ax=ax, color="red", markersize=50)

# plt.title("Share of Census Unit Area Inside 1-Mile Freight Corridor Buffer")
# plt.axis("off")
# plt.tight_layout()
# plt.savefig(FIGURES / "map3_freight_buffer_overlap.png", dpi=300)
# plt.show()

### Map 4: Distance to nearest POE

In [ ]:
# fig, ax = plt.subplots(figsize=(10, 8))

# analysis_gdf.plot(
#     column="dist_to_nearest_poe_mi",
#     legend=True,
#     ax=ax,
#     edgecolor="white",
#     linewidth=0.3,
#     missing_kwds={"color": "lightgray", "label": "Missing"},
# )

# freight_otay["poe_points"].plot(ax=ax, color="red", markersize=60)

# plt.title("Distance to Nearest Port of Entry")
# plt.axis("off")
# plt.tight_layout()
# plt.savefig(FIGURES / "map4_distance_to_poe.png", dpi=300)
# plt.show()

## 16. Optional comparison area

Only run this after the Otay Mesa workflow is complete.

A defensible comparison area should be industrially significant but not directly border-oriented. Possible candidates include Miramar, Kearny Mesa, or Sorrento Valley.

Use the same steps and indicators so that the comparison is methodologically consistent.

In [ ]:
# Example comparison-area selection.

# comparison_query = "miramar|kearny mesa|sorrento"
# comparison_area = planning[
#     planning["name_check"].str.contains(comparison_query, na=False)
# ].copy()

# ax = planning.plot(figsize=(10, 8), facecolor="none", edgecolor="lightgray")
# comparison_area.plot(ax=ax, facecolor="none", edgecolor="purple", linewidth=2)
# plt.title("Comparison Area Candidate")
# plt.axis("off")
# plt.show()

## 17. Optional business-establishment / dynamics extension

Use this only if there is enough time and the data quality is acceptable.

The possible source is a City of San Diego business tax listing or similar business file. A useful file would include:

- business name;
- address;
- NAICS code;
- certificate or start date.

Caution: business addresses may be headquarters or office locations, not actual industrial production sites.

In [ ]:
# businesses_path = DATA_RAW / "business_tax_listing.csv"

# businesses = pd.read_csv(businesses_path)
# display(businesses.head())
# print(businesses.columns.tolist())

# Example NAICS filtering for manufacturing and wholesale sectors.
# businesses["naics"] = businesses["naics"].astype(str)
# industrial_businesses = businesses[
#     businesses["naics"].str.startswith(("31", "32", "33", "42"), na=False)
# ].copy()

# If certificate dates are available:
# industrial_businesses["certificate_date"] = pd.to_datetime(
#     industrial_businesses["certificate_date"],
#     errors="coerce"
# )
# industrial_businesses["certificate_year"] = industrial_businesses["certificate_date"].dt.year

# annual_counts = industrial_businesses.groupby("certificate_year").size()
# annual_counts.plot(figsize=(8, 4))
# plt.title("Industrial Business Certificates Over Time")
# plt.xlabel("Certificate Year")
# plt.ylabel("Number of Businesses")
# plt.tight_layout()
# plt.savefig(FIGURES / "industrial_business_certificates_over_time.png", dpi=300)
# plt.show()

## 18. Interpretation and limitations

Use cautious language. The project identifies spatial association, proximity, overlap, and relative concentration. It does not prove that freight infrastructure caused industrial development.

Suggested interpretation:

> The results suggest a structured spatial association between industrial activity and border-serving freight infrastructure in Otay Mesa.

Avoid:

> Freight infrastructure caused industrial development in Otay Mesa.

Key limitations:

- GeoEnrichment variables are summarized by census units and do not identify exact business locations.
- Business-listing addresses may represent offices or headquarters.
- Zoning and General Plan categories are administrative classifications, not direct measurements of actual industrial activity.
- Distance and buffer indicators measure spatial proximity, not causal influence.
- Comparison-area results should be interpreted only after the Otay Mesa base workflow is complete and checked.